
# Notebook 2e: PACE-OCI & VIIRS Two-Stage Compositing
## Daily → 8-Day → 32-Day Composites with MODIS-Compatible Output

**Key Changes from original 2d:**
1. Output: Individual .bin files (matching MODIS structure)
2. Grid: 600×600 pixels (matching MODIS 2km tiles)
3. Wavelengths: 84 selected (not all 123)
4. Aggregated bands: Band_1-7 pre-computed
5. Thermal: Band31 from VIIRS

**Just configure Cell 1, then "Run All Cells"**

mjf 07/02/2026 (modified for MODIS-compatible output)


In [ ]:
# ## Cell 1: CONFIGURATION

# %%
from pathlib import Path

# =============================================================================
# CONFIGURATION - EDIT THESE VALUES
# =============================================================================

# Input directories
DATA_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF")
PACE_SFREFL_DIR = DATA_DIR / "PACE_SFREFL_DAILY"
VIIRS_VNP21A1D_DIR = DATA_DIR / "VIIRS_VNP21A1D"

# Output base directory
OUTPUT_BASE_DIR = DATA_DIR / "output"

# Year and composite configuration
YEAR = 2025
START_DOY = 65  # March 6 (MODIS VCF convention)
N_COMPOSITES = 12  # Number of 32-day composites
COMPOSITE_DAYS = 32  # Days per composite (4 × 8-day periods)

# Tiles to process
TILES_TO_PROCESS =  [  # 17 tiles - Americas
        "h08v04", "h08v05", "h09v04", "h09v05", "h10v04", "h10v05", "h10v06",
        "h11v02", "h11v03", "h11v04", "h11v05", "h11v08", "h11v09", "h11v10",
        "h12v01", "h12v02", "h12v03",
    # 16 tiles - Americas/Europe
        "h12v04", "h12v05", "h12v09", "h12v10", "h12v12",
        "h13v01", "h13v02", "h13v10", "h13v11", "h13v12",
        "h16v01", "h17v05", "h18v03", "h18v04", "h18v07", "h19v04",
    # 2: [  # 16 tiles - Africa/Asia
        "h19v07", "h19v08", "h19v09", "h19v10", "h19v11", "h19v12",
        "h20v02", "h20v03", "h20v04", "h20v06", "h20v08", "h20v09", 
        "h20v10", "h20v11", "h21v01", "h21v02",   
    # 3: [  # 20 tiles - Asia/Australia
        "h21v04", "h21v05", "h21v06", "h21v10",
        "h22v03", "h22v04", "h23v02", "h23v03",
        "h24v02", "h24v03", "h24v04", "h26v06",
        "h27v04", "h27v06", "h27v07", "h28v11", 
        "h29v11", "h29v12", "h30v12", "h31v11"
    ]

# No-data value and scale factor
NO_DATA = -10001
SCALE_FACTOR = 10000

# Tile size at 2km resolution (MODIS standard)
TILE_SIZE = 600

# MODIS sinusoidal projection parameters
MODIS_SPHERE_RADIUS = 6371007.181
MODIS_TILE_SIZE_M = 1111950.5196666666
MODIS_UPPER_LEFT_X = -20015109.354
MODIS_UPPER_LEFT_Y = 10007554.677

print("Configuration loaded")
print(f"  PACE input: {PACE_SFREFL_DIR}")
print(f"  VIIRS input: {VIIRS_VNP21A1D_DIR}")
print(f"  Output: {OUTPUT_BASE_DIR}")
print(f"  Tiles: {TILES_TO_PROCESS}")
print(f"  Year: {YEAR}, Start DOY: {START_DOY}")
print(f"  Composites: {N_COMPOSITES} × {COMPOSITE_DAYS} days")



In [ ]:
# ## Cell 2: Wavelength Definitions

# 84 vegetation-focused wavelengths to save
WAVELENGTHS_TO_SAVE = [
    # Blue extended (7)
    450, 455, 460, 465, 470, 475, 480,
    # Green/PRI region (8)
    505, 510, 515, 520, 525, 530, 535, 540,
    # Green extended (9)
    545, 550, 555, 560, 565, 570, 575, 580, 586,
    # Red extended (23)
    615, 620, 625, 630, 635, 640, 642, 645, 647, 650, 652,
    655, 657, 660, 662, 665, 667, 670, 672, 675, 677, 679, 682,
    # Red edge (19)
    697, 699, 702, 704, 707, 709, 712, 714, 719, 724, 729, 734, 739, 742, 744, 747, 749, 752, 754,
    # NIR extended (13)
    835, 840, 845, 850, 855, 860, 865, 870, 875, 880, 885, 890, 895,
    # SWIR (5)
    1038, 1249, 1618, 2131, 2258
]

# MODIS band aggregation definitions
MODIS_BAND_RANGES = {
    'Band_1': (620, 670),   # Red
    'Band_2': (841, 876),   # NIR  
    'Band_3': (459, 479),   # Blue
    'Band_4': (545, 565),   # Green
    'Band_5': (1230, 1250), # SWIR1
    'Band_6': (1610, 1652), # SWIR2 - EXPANDED to include 1618
    'Band_7': (2105, 2155), # SWIR3
}

print(f"Wavelengths to save: {len(WAVELENGTHS_TO_SAVE)}")
print(f"MODIS bands to aggregate: {list(MODIS_BAND_RANGES.keys())}")


In [ ]:
# ## Cell 3: Imports


import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional
import logging
from osgeo import gdal
import netCDF4 as nc
from scipy.interpolate import RegularGridInterpolator
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("Imports complete")

In [ ]:

# ## Cell 4: Projection Functions


def get_tile_bounds_sinusoidal(tile: str) -> Tuple[float, float, float, float]:
    """Get tile bounds in sinusoidal coordinates (min_x, min_y, max_x, max_y)."""
    h = int(tile[1:3])
    v = int(tile[4:6])
    
    min_x = MODIS_UPPER_LEFT_X + h * MODIS_TILE_SIZE_M
    max_x = min_x + MODIS_TILE_SIZE_M
    max_y = MODIS_UPPER_LEFT_Y - v * MODIS_TILE_SIZE_M
    min_y = max_y - MODIS_TILE_SIZE_M
    
    return (min_x, min_y, max_x, max_y)

def sinusoidal_to_latlon(x: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Convert sinusoidal coordinates to lat/lon."""
    lat = np.degrees(y / MODIS_SPHERE_RADIUS)
    lon = np.degrees(x / (MODIS_SPHERE_RADIUS * np.cos(np.radians(lat))))
    return lat, lon

def get_tile_latlon_bounds(tile: str) -> Tuple[float, float, float, float]:
    """Get approximate lat/lon bounds for a tile (with buffer)."""
    min_x, min_y, max_x, max_y = get_tile_bounds_sinusoidal(tile)
    
    corners_x = [min_x, max_x, min_x, max_x]
    corners_y = [min_y, min_y, max_y, max_y]
    
    lats, lons = [], []
    for x, y in zip(corners_x, corners_y):
        lat, lon = sinusoidal_to_latlon(x, y)
        lats.append(lat)
        lons.append(lon)
    
    buffer = 1.0
    return min(lats) - buffer, max(lats) + buffer, min(lons) - buffer, max(lons) + buffer

def get_tile_pixel_coords(tile: str) -> Tuple[np.ndarray, np.ndarray]:
    """Get lat/lon coordinates for each pixel center in a 600×600 tile."""
    min_x, min_y, max_x, max_y = get_tile_bounds_sinusoidal(tile)
    
    pixel_size = MODIS_TILE_SIZE_M / TILE_SIZE
    
    x = np.linspace(min_x + pixel_size/2, max_x - pixel_size/2, TILE_SIZE)
    y = np.linspace(max_y - pixel_size/2, min_y + pixel_size/2, TILE_SIZE)
    
    xx, yy = np.meshgrid(x, y)
    lat, lon = sinusoidal_to_latlon(xx, yy)
    
    return lat, lon

print("Projection functions defined")

In [ ]:
# ## Cell 5: File Discovery Functions

def doy_to_date(year: int, doy: int) -> datetime:
    """Convert year and day-of-year to datetime."""
    return datetime(year, 1, 1) + timedelta(days=doy - 1)

def find_viirs_daily_files(input_dir: Path, tile: str, 
                           year: int, start_doy: int, num_days: int) -> Dict[str, Path]:
    """Find VNP21A1D daily files for a tile and date range."""
    files = {}
    days_in_year = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365
    
    for i in range(num_days):
        doy = start_doy + i
        current_year = year
        
        if doy > days_in_year:
            current_year = year + 1
            doy = doy - days_in_year
        
        # VNP21A1D.A2025065.h12v09.002.*.h5
        pattern = f"VNP21A1D.A{current_year}{doy:03d}.{tile}.*.h5"
        matches = list(input_dir.glob(pattern))
        
        if matches:
            files[f"{current_year}{doy:03d}"] = matches[0]
    
    return files

def find_pace_daily_files(input_dir: Path, 
                          year: int, start_doy: int, num_days: int) -> Dict[str, Path]:
    """Find PACE SFREFL L3m daily files for a date range."""
    files = {}
    days_in_year = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365
    
    for i in range(num_days):
        doy = start_doy + i
        current_year = year
        
        if doy > days_in_year:
            current_year = year + 1
            doy = doy - days_in_year
        
        # Convert DOY to YYYYMMDD
        date = datetime(current_year, 1, 1) + timedelta(days=doy - 1)
        date_str = date.strftime('%Y%m%d')
        
        # PACE_OCI.20250306.L3m.DAY.SFREFL.V3_1.rhos.2km.nc
        pattern = f"PACE_OCI.{date_str}.L3m.DAY.SFREFL.*.rhos.*.nc"
        matches = list(input_dir.glob(pattern))
        
        if matches:
            files[f"{current_year}{doy:03d}"] = matches[0]
    
    return files

print("File discovery functions defined")


In [ ]:
# ## Cell 6: Data Reading Functions

def read_viirs_daily(filepath: Path) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Read VNP21A1D daily LST with QA filtering.
    Returns (lst_data, qa_mask) or (None, None) on error.
    """
    try:
        ds = gdal.Open(str(filepath))
        if ds is None:
            return None, None
        
        subdatasets = ds.GetSubDatasets()
        
        # Subdatasets: LST_1KM(3), QC(4)
        lst_ds = gdal.Open(subdatasets[3][0])
        qc_ds = gdal.Open(subdatasets[4][0])
        
        lst_data = lst_ds.ReadAsArray().astype(np.float32)
        qc_data = qc_ds.ReadAsArray()
        
        lst_ds = None
        qc_ds = None
        ds = None
        
        # Apply scaling (0.02 K per DN)
        lst_data = np.where(lst_data == 0, np.nan, lst_data * 0.02)
        
        # QA filtering for VNP21A1D (16-bit)
        modland_qa = qc_data & 0b11
        cloud_flag = (qc_data >> 4) & 0b11
        lst_accuracy = (qc_data >> 14) & 0b11
        
        # Keep: good quality, cloud-free, good LST accuracy
        good_mask = (modland_qa <= 1) & (cloud_flag == 0) & (lst_accuracy >= 2)
        
        lst_data[~good_mask] = np.nan
        lst_data[(lst_data < 150) | (lst_data > 400)] = np.nan
        
        return lst_data, good_mask
        
    except Exception as e:
        logger.warning(f"Error reading VIIRS {filepath.name}: {e}")
        return None, None

def read_pace_daily_for_tile(filepath: Path, tile: str, 
                              wavelengths: List[int]) -> Optional[Dict]:
    """
    Read PACE L3m daily and extract tile region with selected wavelengths.
    """
    try:
        with nc.Dataset(str(filepath), 'r') as ds:
            # Get lat/lon and wavelength arrays
            lat = ds.variables['lat'][:]
            lon = ds.variables['lon'][:]
            file_wavelengths = ds.variables['wavelength'][:]
            
            # Get tile bounds
            lat_min, lat_max, lon_min, lon_max = get_tile_latlon_bounds(tile)
            
            # Find indices for spatial subset
            lat_idx = np.where((lat >= lat_min) & (lat <= lat_max))[0]
            lon_idx = np.where((lon >= lon_min) & (lon <= lon_max))[0]
            
            if len(lat_idx) == 0 or len(lon_idx) == 0:
                return None
            
            # Find wavelength indices
            wl_indices = []
            actual_wavelengths = []
            for wl in wavelengths:
                diffs = np.abs(file_wavelengths - wl)
                min_idx = np.argmin(diffs)
                if diffs[min_idx] <= 3:  # Within 3nm
                    wl_indices.append(min_idx)
                    actual_wavelengths.append(wl)
            
            if not wl_indices:
                return None
            
            # Read rhos subset: (lat, lon, wavelength)
            rhos = ds.variables['rhos'][
                lat_idx[0]:lat_idx[-1]+1,
                lon_idx[0]:lon_idx[-1]+1,
                wl_indices
            ]
            
            # Transpose to (wavelength, lat, lon)
            rhos = np.moveaxis(rhos, -1, 0)
            
            # Handle fill values
            rhos = np.where(rhos < -32000, np.nan, rhos)
            
            return {
                'rhos': rhos,
                'lat': lat[lat_idx],
                'lon': lon[lon_idx],
                'wavelengths': np.array(actual_wavelengths),
            }
            
    except Exception as e:
        logger.warning(f"Error reading PACE {filepath.name}: {e}")
        return None

print("Data reading functions defined")

In [ ]:

# ## Cell 7: Resampling Functions

def resample_to_tile_grid(data: np.ndarray, data_lat: np.ndarray, 
                          data_lon: np.ndarray, target_lat: np.ndarray,
                          target_lon: np.ndarray) -> np.ndarray:
    """Resample data to tile grid using bilinear interpolation."""
    
    if data.ndim == 2:
        # Single band
        try:
            interp = RegularGridInterpolator(
                (data_lat[::-1], data_lon),
                data[::-1, :],
                method='linear',
                bounds_error=False,
                fill_value=np.nan
            )
            points = np.stack([target_lat.ravel(), target_lon.ravel()], axis=-1)
            return interp(points).reshape(target_lat.shape)
        except:
            return np.full(target_lat.shape, np.nan)
    
    elif data.ndim == 3:
        # Multiple bands (wavelength, lat, lon)
        n_bands = data.shape[0]
        resampled = np.full((n_bands, target_lat.shape[0], target_lat.shape[1]), np.nan)
        
        for i in range(n_bands):
            try:
                interp = RegularGridInterpolator(
                    (data_lat[::-1], data_lon),
                    data[i, ::-1, :],
                    method='linear',
                    bounds_error=False,
                    fill_value=np.nan
                )
                points = np.stack([target_lat.ravel(), target_lon.ravel()], axis=-1)
                resampled[i] = interp(points).reshape(target_lat.shape)
            except:
                pass
        
        return resampled
    
    return np.full(target_lat.shape, np.nan)

def resample_viirs_to_2km(data: np.ndarray) -> np.ndarray:
    """Resample VIIRS 1km (1200×1200) to 2km (600×600) by averaging 2×2 blocks."""
    if data.shape != (1200, 1200):
        logger.warning(f"Unexpected VIIRS shape: {data.shape}")
        return np.full((TILE_SIZE, TILE_SIZE), np.nan)
    
    reshaped = data.reshape(TILE_SIZE, 2, TILE_SIZE, 2)
    with np.errstate(all='ignore'):
        return np.nanmean(reshaped, axis=(1, 3))

print("Resampling functions defined")

In [ ]:
# ## Cell 8: Band Aggregation

def aggregate_to_modis_bands(wavelength_data: Dict[int, np.ndarray]) -> Dict[str, np.ndarray]:
    """Aggregate individual wavelengths to MODIS-equivalent bands."""
    aggregated = {}
    
    for band_name, (wl_min, wl_max) in MODIS_BAND_RANGES.items():
        matching_wls = [wl for wl in wavelength_data.keys() if wl_min <= wl <= wl_max]
        
        if not matching_wls:
            logger.debug(f"No wavelengths for {band_name} ({wl_min}-{wl_max}nm)")
            aggregated[band_name] = np.full((TILE_SIZE, TILE_SIZE), np.nan)
            continue
        
        stack = np.stack([wavelength_data[wl] for wl in matching_wls], axis=0)
        with np.errstate(all='ignore'):
            aggregated[band_name] = np.nanmean(stack, axis=0)
    
    return aggregated

print("Band aggregation function defined")

In [ ]:

# ## Cell 9: Output Writing Functions

def save_wavelength_bin(data: np.ndarray, output_dir: Path, 
                        tile: str, doy_key: str, wavelength: int):
    """Save a single wavelength as .bin file."""
    wl_dir = output_dir / "wavelengths"
    wl_dir.mkdir(parents=True, exist_ok=True)
    
    filename = f"PACE-{tile}-{doy_key}-wl_{wavelength:04d}.bin"
    filepath = wl_dir / filename
    
    scaled = np.where(np.isnan(data), NO_DATA, data * SCALE_FACTOR)
    scaled = np.clip(scaled, -32768, 32767).astype(np.int16)
    scaled.tofile(filepath)

def save_aggregated_bin(data: np.ndarray, output_dir: Path,
                        tile: str, doy_key: str, band_name: str):
    """Save an aggregated band as .bin file."""
    agg_dir = output_dir / "aggregated"
    agg_dir.mkdir(parents=True, exist_ok=True)
    
    filename = f"PACE-{tile}-{doy_key}-{band_name}.bin"
    filepath = agg_dir / filename
    
    scaled = np.where(np.isnan(data), NO_DATA, data * SCALE_FACTOR)
    scaled = np.clip(scaled, -32768, 32767).astype(np.int16)
    scaled.tofile(filepath)

def save_thermal_bin(data: np.ndarray, output_dir: Path,
                     tile: str, doy_key: str):
    """Save thermal band as .bin file (raw Kelvin, no scaling)."""
    therm_dir = output_dir / "thermal"
    therm_dir.mkdir(parents=True, exist_ok=True)
    
    filename = f"PACE-{tile}-{doy_key}-Band31.bin"
    filepath = therm_dir / filename
    
    # Thermal stored as raw Kelvin
    thermal_out = np.where(np.isnan(data), NO_DATA, data)
    thermal_out = np.clip(thermal_out, -32768, 32767).astype(np.int16)
    thermal_out.tofile(filepath)

print("Output writing functions defined")

In [ ]:

# ## Cell 10: Compositing Functions

def create_8day_composite(daily_data: List[np.ndarray]) -> np.ndarray:
    """Create 8-day composite from daily data using temporal mean."""
    if not daily_data:
        return None
    
    stack = np.stack(daily_data, axis=0)
    with np.errstate(all='ignore'):
        return np.nanmean(stack, axis=0)

def create_32day_composite(composites_8day: List[np.ndarray]) -> np.ndarray:
    """Create 32-day composite from 4 8-day composites."""
    valid = [c for c in composites_8day if c is not None]
    if not valid:
        return None
    
    stack = np.stack(valid, axis=0)
    with np.errstate(all='ignore'):
        return np.nanmean(stack, axis=0)

print("Compositing functions defined")

In [ ]:

# ## Cell 11: Main Processing Function

def process_tile(tile: str, year: int, start_doy: int, n_composites: int):
    """Process all composites for a single tile."""
    
    print(f"\n{'='*60}")
    print(f"PROCESSING TILE: {tile}")
    print(f"{'='*60}")
    
    # Get target coordinates
    target_lat, target_lon = get_tile_pixel_coords(tile)
    
    # Output directory
    output_dir = OUTPUT_BASE_DIR / tile / str(year) / "2-Composites"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Process each 32-day composite
    for comp_num in range(n_composites):
        comp_start_doy = start_doy + (comp_num * COMPOSITE_DAYS)
        
        # Handle year rollover
        comp_year = year
        if comp_start_doy > 365:
            comp_year = year + 1
            comp_start_doy = comp_start_doy - 365
        
        doy_key = f"{comp_year}{comp_start_doy:03d}"
        
        print(f"\n  Composite {comp_num + 1}/{n_composites}: {doy_key}")
        
        # Storage for 8-day composites
        wavelength_8day_composites = {wl: [] for wl in WAVELENGTHS_TO_SAVE}
        thermal_8day_composites = []
        
        # Process 4 × 8-day periods
        for period in range(4):
            period_start_doy = comp_start_doy + (period * 8)
            period_year = comp_year
            
            if period_start_doy > 365:
                period_year = comp_year + 1
                period_start_doy = period_start_doy - 365
            
            # Find daily files
            viirs_files = find_viirs_daily_files(VIIRS_VNP21A1D_DIR, tile, period_year, period_start_doy, 8)
            pace_files = find_pace_daily_files(PACE_SFREFL_DIR, period_year, period_start_doy, 8)
            
            logger.info(f"    8-day {period+1}/4: VIIRS={len(viirs_files)}, PACE={len(pace_files)} files")
            
            # Collect daily data
            wavelength_daily = {wl: [] for wl in WAVELENGTHS_TO_SAVE}
            thermal_daily = []
            
            # Process PACE daily files
            for doy_str, pace_file in pace_files.items():
                pace_data = read_pace_daily_for_tile(pace_file, tile, WAVELENGTHS_TO_SAVE)
                if pace_data is None:
                    continue
                
                resampled = resample_to_tile_grid(
                    pace_data['rhos'],
                    pace_data['lat'],
                    pace_data['lon'],
                    target_lat,
                    target_lon
                )
                
                for i, wl in enumerate(pace_data['wavelengths']):
                    if wl in wavelength_daily:
                        wavelength_daily[wl].append(resampled[i])
            
            # Process VIIRS daily files
            for doy_str, viirs_file in viirs_files.items():
                lst_data, _ = read_viirs_daily(viirs_file)
                if lst_data is None:
                    continue
                
                resampled_lst = resample_viirs_to_2km(lst_data)
                thermal_daily.append(resampled_lst)
            
            # Create 8-day composites
            for wl in WAVELENGTHS_TO_SAVE:
                if wavelength_daily[wl]:
                    comp_8day = create_8day_composite(wavelength_daily[wl])
                    wavelength_8day_composites[wl].append(comp_8day)
            
            if thermal_daily:
                thermal_8day = create_8day_composite(thermal_daily)
                thermal_8day_composites.append(thermal_8day)
        
        # Create 32-day composites
        wavelength_32day = {}
        for wl in WAVELENGTHS_TO_SAVE:
            if wavelength_8day_composites[wl]:
                wavelength_32day[wl] = create_32day_composite(wavelength_8day_composites[wl])
            else:
                wavelength_32day[wl] = np.full((TILE_SIZE, TILE_SIZE), np.nan)
        
        thermal_32day = create_32day_composite(thermal_8day_composites)
        if thermal_32day is None:
            thermal_32day = np.full((TILE_SIZE, TILE_SIZE), np.nan)
        
        # Aggregate to MODIS bands
        aggregated_32day = aggregate_to_modis_bands(wavelength_32day)
        
        # Save outputs
        # Wavelengths
        for wl, data in wavelength_32day.items():
            if data is not None:
                save_wavelength_bin(data, output_dir, tile, doy_key, wl)
        
        # Aggregated bands
        for band_name, data in aggregated_32day.items():
            save_aggregated_bin(data, output_dir, tile, doy_key, band_name)
        
        # Thermal
        save_thermal_bin(thermal_32day, output_dir, tile, doy_key)
        
        # Report coverage
        wl_valid = sum(1 for wl, d in wavelength_32day.items() if d is not None and np.sum(~np.isnan(d)) > 0)
        thermal_pct = 100 * np.sum(~np.isnan(thermal_32day)) / thermal_32day.size
        print(f"    → Saved: {wl_valid}/{len(WAVELENGTHS_TO_SAVE)} wavelengths, {thermal_pct:.1f}% thermal")
    
    print(f"\n✓ Completed {tile}")


In [ ]:

# ## Cell 12: Run Processing

print("="*60)
print("PACE-VCF COMPOSITE GENERATION")
print("="*60)
print(f"Output format: Individual .bin files (MODIS-compatible)")
print(f"Grid size: {TILE_SIZE}×{TILE_SIZE}")
print(f"Wavelengths: {len(WAVELENGTHS_TO_SAVE)}")
print(f"Aggregated bands: {len(MODIS_BAND_RANGES)}")

for tile in TILES_TO_PROCESS:
    process_tile(tile, YEAR, START_DOY, N_COMPOSITES)

# %% [markdown]
# ## Cell 13: Verification

# %%
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

for tile in TILES_TO_PROCESS:
    comp_dir = OUTPUT_BASE_DIR / tile / str(YEAR) / "2-Composites"
    
    if not comp_dir.exists():
        print(f"\n{tile}: Not found")
        continue
    
    print(f"\n{tile}:")
    
    wl_files = list((comp_dir / "wavelengths").glob("*.bin")) if (comp_dir / "wavelengths").exists() else []
    agg_files = list((comp_dir / "aggregated").glob("*.bin")) if (comp_dir / "aggregated").exists() else []
    therm_files = list((comp_dir / "thermal").glob("*.bin")) if (comp_dir / "thermal").exists() else []
    
    expected_wl = len(WAVELENGTHS_TO_SAVE) * N_COMPOSITES
    expected_agg = len(MODIS_BAND_RANGES) * N_COMPOSITES
    expected_therm = N_COMPOSITES
    
    print(f"  Wavelengths: {len(wl_files)}/{expected_wl}")
    print(f"  Aggregated:  {len(agg_files)}/{expected_agg}")
    print(f"  Thermal:     {len(therm_files)}/{expected_therm}")
    
    # Sample file check
    if wl_files:
        sample = wl_files[0]
        data = np.fromfile(sample, dtype=np.int16).reshape(TILE_SIZE, TILE_SIZE)
        data_float = np.where(data == NO_DATA, np.nan, data / SCALE_FACTOR)
        valid_pct = 100 * np.sum(~np.isnan(data_float)) / data_float.size
        print(f"  Sample ({sample.name}): {valid_pct:.1f}% valid")

print(f"""
Output Structure:
-----------------
{{tile}}/{{year}}/2-Composites/
├── wavelengths/  ({len(WAVELENGTHS_TO_SAVE)} × {N_COMPOSITES} = {len(WAVELENGTHS_TO_SAVE) * N_COMPOSITES} files)
│   └── PACE-{{tile}}-{{doy}}-wl_{{wavelength}}.bin
├── aggregated/   ({len(MODIS_BAND_RANGES)} × {N_COMPOSITES} = {len(MODIS_BAND_RANGES) * N_COMPOSITES} files)
│   └── PACE-{{tile}}-{{doy}}-Band_{{N}}.bin
└── thermal/      ({N_COMPOSITES} files)
    └── PACE-{{tile}}-{{doy}}-Band31.bin

Total files per tile: {len(WAVELENGTHS_TO_SAVE) * N_COMPOSITES + len(MODIS_BAND_RANGES) * N_COMPOSITES + N_COMPOSITES}
""")